# AEM4L2 E03 - Tokenización desde cero

**Objetivo:** entender cómo un sistema divide texto en unidades antes de evaluar o generar.

La tokenización aparece en ASR, NLP y LLMs. En esta clase importa porque afecta cómo el modelo representa palabras y cómo nosotros contamos errores.

**Python exercise relacionado:** `python_puro/AEM4_python_exercises/AEM4L2_audio_pipelines/e05_wer_error_critico.py`


## Qué es tokenización

Definición simple:

> Tokenizar es dividir texto continuo en piezas más pequeñas llamadas tokens.

Un token puede ser:

- una palabra completa;
- un carácter;
- una sub-palabra;
- un símbolo;
- una combinación de partes frecuentes.

No es un detalle menor: define qué unidades puede reconocer, comparar o generar el sistema.


In [ ]:
text = "transferir $99.50 a mi cuenta de ahorros"
print(text)


## Tipo 1: tokenización por palabra

Divide por palabras completas.

```text
transferir | $99.50 | a | mi | cuenta | de | ahorros
```

Ventaja: es fácil de entender.

Problema: si aparece una palabra fuera del vocabulario, el sistema puede fallar.

Ejemplos difíciles:

- nombres propios raros;
- montos;
- códigos;
- medicamentos;
- jerga técnica.


In [ ]:
word_tokens = text.split()
print(word_tokens)
print('cantidad:', len(word_tokens))


## Tipo 2: tokenización por carácter

Divide letra por letra.

Ventaja: puede representar cualquier palabra.

Problema: genera muchos pasos y aumenta riesgo de errores de spelling.


In [ ]:
char_tokens = list(text.replace(' ', '_'))
print(char_tokens[:40])
print('cantidad:', len(char_tokens))


## Tipo 3: tokenización por sub-palabra

Divide palabras en partes frecuentes.

Ejemplo conceptual:

```text
transferir -> trans | fer | ir
ahorros    -> ahorro | s
$99.50     -> $ | 99 | . | 50
```

Ventaja: combina flexibilidad y eficiencia.

Esta es la idea detrás de enfoques como **BPE** y **WordPiece**.


In [ ]:
# Tokenizador subword didáctico, no productivo.
def toy_subword_tokenize(word: str) -> list[str]:
    pieces = {
        'transferir': ['trans', 'fer', 'ir'],
        '$99.50': ['$', '99', '.', '50'],
        'ahorros': ['ahorro', 's'],
    }
    return pieces.get(word, [word])

subword_tokens = []
for word in text.split():
    subword_tokens.extend(toy_subword_tokenize(word))

print(subword_tokens)
print('cantidad:', len(subword_tokens))


## BPE en palabras simples

**BPE (Byte Pair Encoding)** empieza con piezas pequeñas y va fusionando pares frecuentes.

Ejemplo conceptual:

```text
u n b e l i e v a b l e
un | believ | able
```

Si aparece una palabra nueva como `believathon`, puede usar piezas conocidas:

```text
believ | athon
```

Idea clave: BPE aprende pedazos útiles del lenguaje.


## WordPiece en palabras simples

**WordPiece** también usa sub-palabras, pero elige piezas que hacen más probable explicar el corpus.

Ejemplo conceptual:

```text
internationalization -> inter | ##national | ##ization
```

El `##` suele indicar que esa pieza continúa una palabra.

Idea clave: WordPiece intenta construir palabras desde piezas estadísticamente útiles.


## Comparación visual

| Esquema | Flexibilidad | Eficiencia | Riesgo típico |
|---|---:|---:|---|
| Palabra | baja | alta | falla con palabras nuevas |
| Carácter | alta | baja | muchos pasos y typos |
| Sub-palabra | alta | media/alta | segmentación rara pero manejable |

Frase docente: **subword suele ser el punto medio práctico.**


In [ ]:
rows = [
    ('palabra', len(word_tokens)),
    ('carácter', len(char_tokens)),
    ('sub-palabra', len(subword_tokens)),
]

print('Cantidad de tokens según estrategia:')
for name, count in rows:
    bar = '█' * min(count, 50)
    print(f'{name:<12} {count:>3} {bar}')


## Cómo afecta a WER

WER cuenta errores a nivel palabra. Pero si el sistema fragmenta o reconstruye mal, puede cambiar cómo se perciben los errores.

Ejemplo:

```text
Referencia: next Thursday
Hypothesis: next Thurs day
```

Un humano entiende `Thurs day`, pero una métrica word-level puede contarlo como más de un error.

Pregunta:

> ¿Un WER más alto siempre significa peor comprensión humana?
